# 第二部分BERT (Encoder-only-model)
---

第二部份「BERT (Encoder-only-model)」，主要會介紹不同 BERT 相關下游任務的程式碼應用，學習目標著重在如何使用 BERT 相關資源（例如網路上別人訓練好的 BERT 模型）來達成 Token classification、Sequence classification 以及 Text Clustering 等等相關文字處理任務。

### 大綱：
1. 套件介紹
2. 資料前處理
3. Token classification<br>
  3.1 NER<br>
4. Sequence classification<br>
  4.1 Sentiment Classification<br>
  4.2 Relation Extraction (RE)<br>
5. Text Clustering<br>
  5.1 BERTopic 基本用法介紹<br>
  5.2 BERTopic 的 Embedding model<br>
  5.3 BERTopic 的 Clustering model<br>
  5.4 套用中文資料<br>
  5.5 使用Representation方法去微調主題表示



### 1. 套件

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/') # 請將這行修改為自己的 google drive 主目錄

# 下載所需檔案(我們的GitHub檔案，包含dict等)
# !git clone https://github.com/wooooodfire/SMA_G2_PRJ1.git
# !cd SMA_G2_PRJ1
os.listdir() # 確認目錄內容

In [ ]:
# 安裝package
!pip install jieba
!pip install sentence_transformers
!pip install ckip_transformers
!pip install bertopic

In [ ]:
import pandas as pd
import re
import numpy as np
from collections import defaultdict
import multiprocessing
import jieba
import matplotlib.pyplot as plt
from matplotlib.font_manager import fontManager
import ast

# 設定字體
fontManager.addfont("./raw_data/SourceHanSansTW-Regular.otf")
plt.rcParams['font.family'] = 'Source Han Sans TW'
plt.rcParams['font.sans-serif'] = ['Source Han Sans TW']
plt.rcParams['axes.unicode_minus'] = False


**Transformers 和 Sentence-transformers （使用 huggingface 模型）**

In [ ]:
from transformers import BertTokenizerFast, AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification, pipeline
from sentence_transformers import SentenceTransformer
from ckip_transformers.nlp import CkipWordSegmenter, CkipPosTagger, CkipNerChunker

**BERTopic套件**

In [ ]:
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans

In [ ]:
import torch
print(torch.cuda.is_available())

### 2. 資料前處理

In [ ]:
# 讀入starlux資料集
origin_data = pd.read_csv('./raw_data/starlux_dataset.csv')

In [ ]:
# 同義詞
SYNONYM_MAP = {
    '台灣人': '台灣', '台中人': '台灣', '台北人': '台灣', '台籍': '台灣', '中部人': '台灣',
    '日式': '日本', '日籍': '日本',
    '中華航空': '華航', '中航': '華航',
    '美國人': '美國', '美籍': '美國',
    '韓籍': '韓國', '韓國人': '韓國',
}

SPECIAL_SYMBOLS = (
    r'[.＂<>:《》+\-=#$%&()*@＃＄％＆＇\(\)\[\]\{\}（）＊＋－／：'
    r'＜＝＞＠［＼］＾＿｀｛｜｝～｟｠｢｣､、〃》「」『』【】〔〕〖〗〘〙〚〛〜〝〞〟〰〾〿'
    r'–—一\'\'‛""„‟…‧﹏★→─]+'
)
SEG_TOKENS = (";", "；", "！", "!", "？", "?", "。", ",", "，")

URL_PATTERN = re.compile(r'https?://\S+|www\.\S+')


def clean_and_segment(article: str, keep_digits: bool = True, keep_alphabets: bool = True) -> list[str]:
    if not isinstance(article, str):
        article = str(article)

    # 1. 移除網址
    article = URL_PATTERN.sub('', article)

    # 2. 統一換行符號（含字面上的 \\n）
    article = re.sub(r'\\n', '\n', article)
    article = re.sub(r'[\r\t]+', '', article)

    # 3. 依換行切段落，過濾空白段落
    paragraphs = [p for p in article.split('\n') if re.sub(r'\s+', '', p)]

    sentences = []
    for para in paragraphs:
        # 4. 同義詞替換（斷句前先統一用詞）
        for old, new in SYNONYM_MAP.items():
            para = para.replace(old, new)

        # 5. 選擇性保留數字 / 英文
        if not keep_digits:
            para = re.sub(r'\d*\.?\d+', '', para)
        if not keep_alphabets:
            para = re.sub(r'[a-zA-Z]+', '', para)

        # 6. 移除空格與特殊符號
        para = re.sub(r'\s+', '', para)
        para = re.sub(SPECIAL_SYMBOLS, '', para)

        # 7. 斷句
        seg_pattern = '|'.join(map(re.escape, SEG_TOKENS))
        segs = re.split(seg_pattern, para)

        # 8. 只保留中文字（對齊下方的 re.sub('[^\u4e00-\u9fff]+', '', x)）
        segs = [re.sub(r'[^\u4e00-\u9fff]', '', s) for s in segs]

        # 9. 過濾：長度 > 1 且含中文
        segs = [s for s in segs
                if len(s.strip()) > 1 and re.search(r'[\u4e00-\u9fff]', s)]
        sentences.extend(segs)

    return sentences


# ── DataFrame 處理 ──────────────────────────────────────────────────────
# 去除一些不需要的欄位
metaData = origin_data.drop(['artPoster'], axis=1)

# 將 artDate 轉換為 datetime 格式，並只保留日期部分
metaData["artDate"] = pd.to_datetime(metaData["artDate"]).dt.date

# 去除文章內容為空值的筆數
metaData.dropna(subset=['artContent'], axis=0, how='any', inplace=True)

# 清理 & 斷句（每個 cell → 句子 list）
metaData['sentence'] = metaData['artContent'].apply(clean_and_segment)

# 展開為每句一列
metaData = metaData.explode('sentence').reset_index(drop=True)

# 過濾斷句後的空值或空字串
metaData = metaData[metaData['sentence'].str.len() > 0].reset_index(drop=True)

metaData.head(10)

In [ ]:
print("=" * 40)
print(f"總句數：{len(metaData)}")
print(f"總文章數：{metaData['system_id'].nunique()}")
print(f"日期範圍：{metaData['artDate'].min()} ～ {metaData['artDate'].max()}")
print(f"句子平均長度：{metaData['sentence'].str.len().mean():.1f} 字")
print(f"句子最短／最長：{metaData['sentence'].str.len().min()} ／ {metaData['sentence'].str.len().max()} 字")
print("=" * 40)
metaData.info()

## 3. Token classification

### NER
使用 Huggingface 上面已經針對 NER 任務 finetune 好的 BERT 模型來實作<br>
Huggingface 的模型列表：https://huggingface.co/models?sort=trending

如果找不到自己適用的模型的話，也可以透過 fine-tune 來建立自己的模型。<br>
本課程因為時間與資源因素，僅針對「如何使用網路上他人 fine-tune 好的模型」進行程式碼示範，不提供 fine-tune 程式碼範例。<br>
如有需要，可參考 Huggingface 相關教學文章：[Fine-tune a pretrained model](https://huggingface.co/docs/transformers/training)

#### 3.1 中文 NER：<br>

**將CKIP套用到我們先前處理好的資料集**

使用 CKIP 開發的 NLP 套件：ckip_transformers<br>
- 使用的 WS 模型：https://huggingface.co/ckiplab/bert-base-chinese-ws<br>
- 使用的 POS 模型：https://huggingface.co/ckiplab/bert-base-chinese-pos<br>
- 使用的 NER 模型：https://huggingface.co/ckiplab/bert-base-chinese-ner

In [ ]:
# # 初始化 ckip 工具 device=0 使用GPU ｜ device=-1 使用CPU（速度會很慢）
# # Mac使用者可以設定 device=torch.device("mps") 使用GPU
# ws_driver  = CkipWordSegmenter(model_name="ckiplab/bert-base-chinese-ws", device=0) # Word Segmenter斷詞
# pos_driver = CkipPosTagger(model_name="ckiplab/bert-base-chinese-pos", device=0) # POS tagger 詞性標記
# ner_driver = CkipNerChunker(model_name="ckiplab/bert-base-chinese-ner", device=0) # NER識別

In [ ]:
# 跑模型
# text = metaData['sentence'].tolist()

# # 執行處理
# ws = ws_driver(text) # 斷詞
# pos = pos_driver(ws) # POS
# ner = ner_driver(text) # NER

# # 將斷詞以及 pos 結果合在一起顯示
# def pack_ws_pos_sentece(sentence_ws, sentence_pos):
#     assert len(sentence_ws) == len(sentence_pos) # 確認斷詞和POS的長度相同
#     res = []
#     for word_ws, word_pos in zip(sentence_ws, sentence_pos):
#         res.append(f"{word_ws}({word_pos})") # 合併在一起
#     return "\u3000".join(res)

# sentences, packed_sentences, entities = [], [], []

# # 儲存結果
# for sentence, sentence_ws, sentence_pos, sentence_ner in zip(text, ws, pos, ner):
#     sentences.append(sentence)
#     packed_sentences.append(pack_ws_pos_sentece(sentence_ws, sentence_pos))
#     clean_entities = [{'word': entity.word, 'ner': entity.ner} for entity in sentence_ner]
#     entities.append(clean_entities)

# # 將結果存在一個 dataframe 中
# ner_results = pd.DataFrame({
#    'sentence': sentences,
#    'packed_sentence': packed_sentences,
#    'entities': entities
# })

# ner_results.head(10)
# # 備份
# ner_results.to_csv('raw_data/ckip_transformers_base.csv', index=False)


In [ ]:
# # 讀取已跑好的資料

# 讀取
ner_results = pd.read_csv('raw_data/ckip_transformers_base.csv')

# entities 欄位從字串還原為 list of dict
ner_results['entities'] = ner_results['entities'].apply(ast.literal_eval)

In [ ]:
def plot_top10(df, title, color='steelblue'):
    """
    df: 兩欄 DataFrame，第一欄為標籤，第二欄為數量
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    labels = df.iloc[:, 0].astype(str)
    counts = df.iloc[:, 1]

    bars = ax.barh(labels[::-1], counts[::-1], color=color)  # 由高到低排列

    # 在每個 bar 右側標上數值
    for bar, count in zip(bars, counts[::-1]):
        ax.text(bar.get_width() + counts.max() * 0.01, bar.get_y() + bar.get_height() / 2,
                str(count), va='center', fontsize=10)

    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Count', fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
# 1. NER Frequency（ TOP10）
from collections import Counter

# 攤平所有 entity，取出 ner 類型
all_ner_types = [
    entity['ner']
    for entities in ner_results['entities']
    for entity in entities
]

ner_freq = pd.DataFrame(
    Counter(all_ner_types).most_common(10),
    columns=['NER_Type', 'Count']
)
print(ner_freq)
plot_top10(ner_freq, 'NER Type Frequency TOP10', color='steelblue')

In [ ]:
# 2. GPE TOP10
def get_ner_top(ner_results, ner_type, top_n=10):
    words = [
        entity['word']
        for entities in ner_results['entities']
        for entity in entities
        if entity['ner'] == ner_type
    ]
    return pd.DataFrame(
        Counter(words).most_common(top_n),
        columns=['Word', 'Count']
    )

gpe_top10    = get_ner_top(ner_results, 'GPE')

print("=== GPE ===");    print(gpe_top10)

plot_top10(gpe_top10, 'GPE (地理位置) TOP10', color='darkorange')

In [ ]:
# 3. NORP TOP10
norp_top10   = get_ner_top(ner_results, 'NORP')

print("=== NORP ===");   print(norp_top10)

plot_top10(norp_top10, 'NORP (國籍/宗教/政治) TOP10', color='seagreen')


In [ ]:
# 4. ORG TOP10
org_top10    = get_ner_top(ner_results, 'ORG')

print("=== ORG ===");    print(org_top10)

plot_top10(org_top10, 'ORG (組織) TOP10', color='mediumpurple')

In [ ]:
# 5. PERSON TOP8

# 排除清單 & 別稱對照
not_person = []
person_alias = {'Joli': '蔡依林', 'jolin': '蔡依林', 'Jolin': '蔡依林',
                '張': '張國煒', '董仔': '張國煒', 'k董': '張國煒', 'K董': '張國煒', '張董': '張國煒'}

# 抽取 → 排除 → 合併別稱 → 計數
person_top10 = (
    pd.Series([e['word'] for entities in ner_results['entities']
                         for e in entities if e['ner'] == 'PERSON'])
      .pipe(lambda s: s[~s.isin(not_person)])
      .replace(person_alias)
      .value_counts()
      .nlargest(8)
      .reset_index()
      .set_axis(['Word', 'Count'], axis=1)
)

print(person_top10)
plot_top10(person_top10, 'PERSON (人名) TOP8', color='tomato')

In [ ]:
# 6. POS Frequency（所有詞性）
all_pos = []
for packed in ner_results['packed_sentence']:
    # 抓出所有 (POS) 括號內的標籤
    pos_tags = re.findall(r'\((\w+)\)', packed)
    all_pos.extend(pos_tags)

pos_freq = pd.DataFrame(
    Counter(all_pos).most_common(10),
    columns=['POS_Tag', 'Count']
)
print(pos_freq)

plot_top10(pos_freq, 'POS Tag Frequency TOP10', color='slategray')



In [ ]:
# 7. Noun TOP10

# 排除清單 & 別稱對照（Noun 專用）
not_noun = []  # 有需要排除的詞可以加在這裡
noun_alias = {'星宇航空': '星宇', '董': '張國煒', '董仔': '張國煒'}

def get_pos_words_top(ner_results, pos_prefix, top_n=10,
                      exclude=None, alias=None):
    words = []
    for packed in ner_results['packed_sentence']:
        pairs = re.findall(r'([^\s　]+)\((\w+)\)', packed)
        for word, pos in pairs:
            if pos.startswith(pos_prefix):
                words.append(word)

    return (pd.Series(words)
              .pipe(lambda s: s[~s.isin(exclude)] if exclude else s)
              .replace(alias if alias else {})
              .value_counts()
              .nlargest(top_n)
              .reset_index()
              .set_axis(['Word', 'Count'], axis=1))


# Noun（Nb 或 Na，依需求傳入）
noun_top10 = get_pos_words_top(ner_results, 'Nb', exclude=not_noun, alias=noun_alias)

print("=== Noun (Nb) ==="); print(noun_top10)
plot_top10(noun_top10, 'Noun / Nb (專有名詞) TOP10', color='goldenrod')

In [ ]:
# 8. VC TOP10

def get_pos_words_top(ner_results, pos_prefix, top_n=10,
                      exclude=None, alias=None, min_len=1):
    words = []
    for packed in ner_results['packed_sentence']:
        pairs = re.findall(r'([^\s　]+)\((\w+)\)', packed)
        for word, pos in pairs:
            if pos.startswith(pos_prefix):
                words.append(word)

    return (pd.Series(words)
              .pipe(lambda s: s[s.map(len) >= min_len] if min_len > 1 else s)  # 長度過濾  # 長度過濾
              .pipe(lambda s: s[~s.isin(exclude)] if exclude else s)
              .replace(alias if alias else {})
              .value_counts()
              .nlargest(top_n)
              .reset_index()
              .set_axis(['Word', 'Count'], axis=1))


# VC：過濾長度為 1 的字
vc_top10 = get_pos_words_top(ner_results, 'VC', min_len=2)

print("=== VC ==="); print(vc_top10)
plot_top10(vc_top10, 'VC (及物動詞) TOP10', color='cadetblue')

來試試看用tiny版本

In [ ]:
# 跑模型
# # 初始化 ckip 工具 device=0 使用GPU ｜ device=-1 使用CPU（速度會很慢）
# # Mac使用者可以設定 device=torch.device("mps") 使用GPU
# ws_driver  = CkipWordSegmenter(model_name="ckiplab/bert-tiny-chinese-ws", device=0) # Word Segmenter斷詞
# pos_driver = CkipPosTagger(model_name="ckiplab/bert-tiny-chinese-pos", device=0) # POS tagger 詞性標記
# ner_driver = CkipNerChunker(model_name="ckiplab/bert-tiny-chinese-ner", device=0) # NER識別


# # 執行處理
# ws = ws_driver(text) # 斷詞
# pos = pos_driver(ws) # POS
# ner = ner_driver(text) # NER

# sentences, packed_sentences, entities = [], [], []

# # 儲存結果
# for sentence, sentence_ws, sentence_pos, sentence_ner in zip(text, ws, pos, ner):
#     sentences.append(sentence)
#     packed_sentences.append(pack_ws_pos_sentece(sentence_ws, sentence_pos))
#     clean_entities = [{'word': entity.word, 'ner': entity.ner} for entity in sentence_ner]
#     entities.append(clean_entities)

# # 將結果存在一個 dataframe 中
# ner_results = pd.DataFrame({
#    'sentence': sentences,
#    'packed_sentence': packed_sentences,
#    'entities': entities
# })

# ner_results.head(10)
# # 備份
# ner_results.to_csv('raw_data/ckip_transformers_tiny.csv', index=False)

In [ ]:
# # 讀取已跑好的資料

# 讀取
ner_results = pd.read_csv('raw_data/ckip_transformers_tiny.csv')

# entities 欄位從字串還原為 list of dict
ner_results['entities'] = ner_results['entities'].apply(ast.literal_eval)

In [ ]:
from collections import Counter

# 攤平所有 entity，取出 ner 類型
all_ner_types = [
    entity['ner']
    for entities in ner_results['entities']
    for entity in entities
]

ner_freq = pd.DataFrame(
    Counter(all_ner_types).most_common(10),
    columns=['NER_Type', 'Count']
)
print(ner_freq)
plot_top10(ner_freq, 'NER Type Frequency TOP10', color='steelblue')

In [ ]:
# 2. GPE TOP10
def get_ner_top(ner_results, ner_type, top_n=10):
    words = [
        entity['word']
        for entities in ner_results['entities']
        for entity in entities
        if entity['ner'] == ner_type
    ]
    return pd.DataFrame(
        Counter(words).most_common(top_n),
        columns=['Word', 'Count']
    )

gpe_top10    = get_ner_top(ner_results, 'GPE')

print("=== GPE ===");    print(gpe_top10)

plot_top10(gpe_top10, 'GPE (地理位置) TOP10', color='darkorange')

In [ ]:
# 3. NORP TOP10
norp_top10   = get_ner_top(ner_results, 'NORP')

print("=== NORP ===");   print(norp_top10)

plot_top10(norp_top10, 'NORP (國籍/宗教/政治) TOP10', color='seagreen')


In [ ]:
# 4. ORG TOP10
org_top10    = get_ner_top(ner_results, 'ORG')

print("=== ORG ===");    print(org_top10)

plot_top10(org_top10, 'ORG (組織) TOP10', color='mediumpurple')

In [ ]:
# 5. PERSON TOP10

# 排除清單 & 別稱對照
not_person = []
person_alias = {'Joli': '蔡依林', 'jolin': '蔡依林', 'Jolin': '蔡依林',
                '張': '張國煒', '董仔': '張國煒', 'k董': '張國煒', 'K董': '張國煒', '張董': '張國煒'}

# 抽取 → 排除 → 合併別稱 → 計數
person_top10 = (
    pd.Series([e['word'] for entities in ner_results['entities']
                         for e in entities if e['ner'] == 'PERSON'])
      .pipe(lambda s: s[~s.isin(not_person)])
      .replace(person_alias)
      .value_counts()
      .nlargest(8)
      .reset_index()
      .set_axis(['Word', 'Count'], axis=1)
)

print(person_top10)
plot_top10(person_top10, 'PERSON (人名) TOP8', color='tomato')

In [ ]:
# 6. POS Frequency（所有詞性）
all_pos = []
for packed in ner_results['packed_sentence']:
    # 抓出所有 (POS) 括號內的標籤
    pos_tags = re.findall(r'\((\w+)\)', packed)
    all_pos.extend(pos_tags)

pos_freq = pd.DataFrame(
    Counter(all_pos).most_common(10),
    columns=['POS_Tag', 'Count']
)
print(pos_freq)

plot_top10(pos_freq, 'POS Tag Frequency TOP10', color='slategray')



In [ ]:
# 7. Noun TOP10

# 排除清單 & 別稱對照（Noun 專用）
not_noun = []  # 有需要排除的詞可以加在這裡
noun_alias = {'星宇航空': '星宇', '董': '張國煒', '董仔': '張國煒'}

def get_pos_words_top(ner_results, pos_prefix, top_n=10,
                      exclude=None, alias=None):
    words = []
    for packed in ner_results['packed_sentence']:
        pairs = re.findall(r'([^\s　]+)\((\w+)\)', packed)
        for word, pos in pairs:
            if pos.startswith(pos_prefix):
                words.append(word)

    return (pd.Series(words)
              .pipe(lambda s: s[~s.isin(exclude)] if exclude else s)
              .replace(alias if alias else {})
              .value_counts()
              .nlargest(top_n)
              .reset_index()
              .set_axis(['Word', 'Count'], axis=1))


# Noun（Nb 或 Na，依需求傳入）
noun_top10 = get_pos_words_top(ner_results, 'Nb', exclude=not_noun, alias=noun_alias)

print("=== Noun (Nb) ==="); print(noun_top10)
plot_top10(noun_top10, 'Noun / Nb (專有名詞) TOP10', color='goldenrod')

In [ ]:
# 8. VC TOP10

def get_pos_words_top(ner_results, pos_prefix, top_n=10,
                      exclude=None, alias=None, min_len=1):
    words = []
    for packed in ner_results['packed_sentence']:
        pairs = re.findall(r'([^\s　]+)\((\w+)\)', packed)
        for word, pos in pairs:
            if pos.startswith(pos_prefix):
                words.append(word)

    return (pd.Series(words)
              .pipe(lambda s: s[s.map(len) >= min_len] if min_len > 1 else s)  # 長度過濾  # 長度過濾
              .pipe(lambda s: s[~s.isin(exclude)] if exclude else s)
              .replace(alias if alias else {})
              .value_counts()
              .nlargest(top_n)
              .reset_index()
              .set_axis(['Word', 'Count'], axis=1))


# VC：過濾長度為 1 的字
vc_top10 = get_pos_words_top(ner_results, 'VC', min_len=2)

print("=== VC ==="); print(vc_top10)
plot_top10(vc_top10, 'VC (及物動詞) TOP10', color='cadetblue')

## 4. Sequence classification

### 4.1 Sentiment Classification
使用 Huggingface 上面已經針對 Sentiment classification 任務 finetune 的 BERT 模型來實作<br>
使用的模型：https://huggingface.co/techthiyanes/chinese_sentiment<br><br>
情緒(start 1到star 5)：<br>
1. Semi-negation<br>
2. Negation<br>
3. Neutral<br>
4. Semi-positive<br>
5. Positive

In [ ]:
# 載入已經被 fine-tune 過的 BERT 模型
model_name = "techthiyanes/chinese_sentiment"  # 你可以將這裡換成你想要使用的模型
model = pipeline('sentiment-analysis', model=model_name)

# 使用模型來進行情緒分析
text = ["我喜歡這部電影！", "他的行為讓我很困擾"]
result = model(text)

# 輸出結果(標籤和分數)
result


In [ ]:
# 建立一個新的 dataframe 來儲存結果
results_df = pd.DataFrame(columns=['sentence', 'label', 'score'])
results_df['sentence'] = metaData['sentence']

# 定義一個函數來進行情緒分析
def analyze_sentiment(sentence):
    result = model([sentence])
    return pd.Series([result[0]['label'], result[0]['score']])

# 使用 apply 函數來進行情緒分析
results_df[['label', 'score']] = metaData['sentence'].apply(analyze_sentiment)

# 輸出結果
results_df.head(10)

同學可以依據前幾週的程式碼，對情緒分析後的句子進行近一步的探索（參考第四周、第五周的情緒）

### 4.2 Relation Extraction (RE)
使用 Huggingface 上面已經針對 RE 任務 finetune 的 BERT 模型來實作<br>
使用的模型：https://huggingface.co/teppei727/bert-large-relation14

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("teppei727/bert-large-relation14")
model = AutoModelForSequenceClassification.from_pretrained("teppei727/bert-large-relation14")
re_model = pipeline("text-classification", model=model, tokenizer=tokenizer)

# 使用模型來進行 RE
text = [
    "My name is Wolfgang and I live in Berlin",
    "Obama was a USA president."
]
result = re_model(text)

# 輸出結果
result

In [ ]:
# 讀入英文示範資料集
en_origin_data = pd.read_csv('./raw_data/en_sentence_chapter.csv')
en_origin_data.head()

# 建立一個新的 dataframe 來儲存結果
re_results_df = pd.DataFrame(columns=['sentence', 'label', 'score'])
re_results_df['sentence'] = en_origin_data['sentence']

# 定義一個函數來進行 RE 任務
def get_re_result(sentence):
    result = re_model([sentence])
    return pd.Series([result[0]['label'], result[0]['score']])

# 使用 apply 函數來進行 RE 任務
re_results_df[['label', 'score']] = en_origin_data['sentence'].apply(get_re_result)

# 輸出結果
re_results_df.head(10)

注意：RE 跟 NER 一樣，不同的模型會有不同的關係型態，尤其是使用特定領域資料 fine-tune 的模型，通常會有該特定領域專有的關係型態

## 5. Text Clustering

以下使用 BERTopic 來實作 BERT-based 的 Text Clustering，並介紹 clustering-based 的主題模型<br>
BERTopic: https://maartengr.github.io/BERTopic/index.html<br><br>

![image.png](static/proj3-bertopic.png)

簡單介紹BERTopic套件的模組架構，每一列的組件代表不同的處理步驟，主要可以分為兩個階段，其中相同顏色的組件代表可替換的選項：<br>
- **Topic Creation 主題分群**
    - **Embeddings**: 使用語言模型將文本句子轉換成向量，用於進行分群
    - **Dimension Reduction**: 把高維度的語意向量降維方便後續處理
    - **Clustering**: 把降維後的語意向量進行分群（主題）
---
- **Topic Representation 主題表示**
    - **Tokenizer**: 將文本句子的詞進行轉換成可計算的表示
    - **Topic Representation**: 從每個主題中抽取出主題的關鍵代表詞
---
除此之外，新版本的BERTopic也提供了方法去進一步微調每個主題表示，可以使用GPT、KEYBERT、Spacy等模型或方法去調整並找出更好的主題模型的代表詞或標籤。

#### 5.1 基本用法介紹

In [ ]:
docs = en_origin_data['sentence'].tolist()

# 定義不同 layer 所要使用的模型與方法
embedding_model = "all-MiniLM-L6-v2" # Embeddings layer
hdbscan_model = HDBSCAN() # Clustering layer
vectorizer_model = CountVectorizer()

topic_model = BERTopic(embedding_model=embedding_model, hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model)
topics, probs = topic_model.fit_transform(docs[:500])

In [ ]:
topic_model.get_topic_info()

In [ ]:
# 列出Topic 6的關鍵字和其對應的c TF-IDF分數
topic_model.get_topic(6)

In [ ]:
# 列出前10筆文章的BERTopic資訊
doc_topic_info = topic_model.get_document_info(docs[:500])
doc_topic_info.head(10)

In [ ]:
# 視覺化主題分布：圓圈大小是主題的大小，圓圈的距離是主題之間的相似度
topic_model.visualize_topics()

#### 5.2 Embedding model<br>
BERTopic 支援多種 embedding 模型與方法，包含基本的 Huggingface 模型，也提供了 LLM-based 的 embedding 可做選擇。<br>
更多請參考文件：https://maartengr.github.io/BERTopic/getting_started/embeddings/embeddings.html#scikit-learn-embeddings

In [ ]:
# 使用 sentence_transformers 相關語言模型作為 embedding_model
sentence_model = SentenceTransformer("google-bert/bert-base-uncased")

# 定義不同 layer 所要使用的模型與方法
hdbscan_model = HDBSCAN()
vectorizer_model = CountVectorizer()

# 將 BERTopic 的 embedding_model 替換為其他模型（sentence_model）
embed_topic_model = BERTopic(embedding_model=sentence_model, hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model)
topics, probs = embed_topic_model.fit_transform(docs[:500])

In [ ]:
import openai
from bertopic.backend import OpenAIBackend

# 也可以使用 OpenAI API 來獲取性能更強大的語言模型作為 embedding_model
client = openai.OpenAI(api_key="請替換為你的 openai key")
openai_embedding_model = OpenAIBackend(client, "text-embedding-ada-002")

# 定義不同 layer 所要使用的模型與方法
hdbscan_model = HDBSCAN()
vectorizer_model = CountVectorizer()

# 將 BERTopic 的 embedding_model 替換為其他模型（openai_embedding_model）
openai_topic_model = BERTopic(embedding_model=openai_embedding_model, hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model)
topics, probs = openai_topic_model.fit_transform(docs[:500])

#### 5.3 Clustering model<br>
參考文件：https://maartengr.github.io/BERTopic/getting_started/clustering/clustering.html

調整 topic modeling 流程中的 clustering 算法

In [ ]:
# 使用 KMeans 作為分群算法，需要指定分群個數
cluster_model = KMeans(n_clusters=10)

# 定義不同 layer 所要使用的模型與方法
embedding_model = "all-MiniLM-L6-v2"
vectorizer_model = CountVectorizer()

# 將 BERTopic 的 hdbscan_model 替換為其他模型（cluster_model）
kmeans_topic_model = BERTopic(embedding_model=embedding_model, hdbscan_model=cluster_model, vectorizer_model=vectorizer_model)
topics, probs = kmeans_topic_model.fit_transform(docs[:500])

kmeans_topic_model.get_topic_info()

查看 text clustering 結果

In [ ]:
topics[:10]

#### 5.4 套用於中文<br>
為了套用到中文文章，各元件必須修改為支援中文的方法，主要針對 embedding model 以及 tokenizer

In [ ]:
# 中文文章
docs_zh = metaData['sentence'].tolist()

# 設定繁體中文詞庫
jieba.set_dictionary('./dict/dict.txt.big')

# 新增 stopwords
with open('./dict/stopwords.txt',encoding="utf-8") as f:
    stopwords = [line.strip() for line in f.readlines()]

# !!!!再看需不需要
# 參數形式手動加入停用字
stopwords_manual = ["知道", "現在", "一堆", "本來", "一直", "看到", "目前", "不用", "直接", "明年", "10", "這種", "一點", "根本", "表示",
                    "一下", "反正", "繼續", "好像", "好了", "去年", "未來", "需要", "使用", "之後", "太多", "今天", "回來", "機會", "增加",
                    "樓上", "感覺", "會不會", "101", "工作", "一年", "選擇", "加入", "更多", "考慮", "聽到", "預計", "339", "空間",
                    "出發", "不可能", "今年", "時間", "經營", "需求", "以後", "能力", "當初", "原文", "不發", "幾年", "至少", "過去", "最後",
                    "有沒有", "還在", "準備", "小時", "以前", "位置", "持續", "提供", "幹嘛", "認為", "標題", "有什麼", "幾個", "看看", "14",
                    "網路", "討論", "重點", "變成", "系統", "兩家", "裡面", "要不要", "只能", "來源", "這麼多", "尤其", "規模", "事情", "完全",
                    "原來", "條件", "最近", "指出", "決定", "一次", "地方", "連結", "每週", "代表", "關係", "發現", "造成", "遇到", "日期", "359",
                    "謝謝", "15", "引述", "相關", "內部", "銘言", "XD","笑死","張國煒","開航","廁所","希望","精品","資方","新聞","落地", "客人",
                    "董仔","下去","那麼多","廣體機","美東","XDD","董事長","廣體", "產業","艙等","東南亞","桃機","每周","TPE","餐點","網友","787","理由",
                    "分享","來回","完成","12","聽說","客運","東西","情況","反串","設計", "羽田","新航線","期間","芬蘭","Lulu","有夠","回程","兩個","每年"]
stopwords.extend(stopwords_manual)

# 設定中文 embedding model
bert_sentence_model = SentenceTransformer("google-bert/bert-base-chinese")

# 將中文文章轉換為 embedding
embeddings = bert_sentence_model.encode(docs_zh, show_progress_bar=True)

# 定義不同 clustering layer 所要使用的模型與方法（就用 default 的 HDBSCAN）
hdbscan_model = HDBSCAN()

# 定義一個適合中文的分詞函數
def tokenize_zh(text):
    words = jieba.lcut(text)
    return words

# 建立一個使用 jieba 分詞的 CountVectorizer
jieba_vectorizer = CountVectorizer(tokenizer=tokenize_zh, stop_words=stopwords, analyzer='word', token_pattern=u"(?u)\\b\\w+\\b")

# 使用 BERTopic 進行主題模型建立
zh_topic_model = BERTopic(embedding_model=bert_sentence_model, vectorizer_model=jieba_vectorizer, verbose=True, top_n_words=30)
topics, probs = zh_topic_model.fit_transform(docs_zh, embeddings)

zh_topic_model.get_topic_info()

In [ ]:
zh_topic_model.visualize_topics()

In [ ]:
# 估算每個文件對BERTopic每個主題的機率分布
topic_distr, _ = zh_topic_model.approximate_distribution(docs_zh)

In [ ]:
# 以第18個文件為例，列出這份文件對每個主題的機率分布
zh_topic_model.visualize_distribution(topic_distr[18])

In [ ]:
# 列出主題的代表詞和其對應的權重
zh_topic_model.get_topic(2)

查看特定文章的主題分佈

In [ ]:
# 視覺化顯示主題-詞彙分佈
topic_n = 2
data = zh_topic_model.get_topic(topic_n)

# 轉換為DataFrame
df = pd.DataFrame(data, columns=['word', 'prob'])
df = df[df['word'] != ' ']

# 根據prob排序並選出前10名
top_10 = df.sort_values('prob', ascending=False).head(10)

# 畫出長條圖
plt.figure(figsize=(10,6))
plt.barh(top_10['word'], top_10['prob'], color='navy')
plt.xlabel('機率')
plt.title(f'主題 {topic_n} 詞彙機率前10名')
plt.gca().invert_yaxis()
plt.show()

#### 5.5 用模型進行主題標籤的調整
新版本的BERTopic提供許多方法供我們調整以c TF-IDF的主題表示，讓我們能得到更能準確描述各個文件集合的字詞去描述每個主題。

目前的Representation方法大致分成兩種：
- 非生成模型：專注於調整或改善每個主題表示的關鍵字
- 生成模型：會根據每個主題的詞語和代表文件，去標記或總結主題

在這邊我們先以KeyBERT非生成模型方法做示範如何去調整BERTopic的主題表示詞語

##### KeyBERT方法說明
---
KeyBERT是一種文本關鍵字提取模型，其方法是透過計算文本中每個N-gram或者token字詞和文件本身之間得BERT embedding的餘弦相似度，去找到文件的關鍵字。
BERTopic根據KeyBERT的方法去建立了一個應用於每個Topic的關鍵字提取流程，架構如下圖所示。

![image.png](static/proj3-keybert.png)

1. 首先針對Topic n，會先依據這個主題的c TF-IDF去找出最能代表主題的幾篇文件，和最能代表主題的幾個候選關鍵代表字。
    - 候選關鍵代表字是根據c TF-IDF的大小去排序，找出前Ｎ個代表字
    - 主題的代表文件挑選方法是比較主題的c TF-IDF和每篇文件的c TF-IDF
2. 接著會利用BERT去產生這些文件和候選代表字的Embedding向量
    - 其中所有代表文件的Embedding向量會取平均作為這個Topic n的向量表示
3. 接著去計算Topic n的向量表示和每個候選代表字的Embedding向量的相似度去找出這個Topic n的最終代表字

根據這個調整後KeyBERT的方法，就能快速的去調整並找出每個主題的代表關鍵字。

In [ ]:
from bertopic.representation import KeyBERTInspired
# KeyBERT
keybert = KeyBERTInspired()

# 設定HDBscan模型
hdbscan_model = HDBSCAN(min_cluster_size=5, min_samples=30)

# 定義我們要用到的representation model（同學如果想比較其他模型可以在這邊加入其他模型方法）
representation_model = {
    "KeyBERT": keybert,
}

In [ ]:
# 建立BERTopic模型
representation_topic_model = BERTopic(
  # Sub-models
  embedding_model=bert_sentence_model,
  vectorizer_model=jieba_vectorizer,
  # 設定Representation model
  representation_model=representation_model,
  # Hyperparameters
  top_n_words=30,
  verbose=True
)

# Train model
topics, probs = representation_topic_model.fit_transform(docs_zh, embeddings)

In [ ]:
# 觀察KeyBERT微調後的主題表示
representation_topic_model.get_topic_info()

主題表示的微調方法除了KeyBERT以外還有MMR、Spacy等多種方法，除此之外也能進一步透過生成模型的方法進行主題的標籤生成，例如：GPT、Llama等，由於教材篇幅以及方法的複雜性原因，這邊另外提供同學們BERTopic的官方網站說明以及Llama示範的colab程式碼。

有興趣的同學可以參考並嘗試替換不同方法去調整主題的代表詞，或者進一步用生成模型對每個生成主題進行標記或者總結，去更好的解釋每個主題的內容。

- BERTopic官網說明：https://maartengr.github.io/BERTopic/getting_started/representation/representation.html
- 使用地端Llama模型進行BERTopic總結：https://colab.research.google.com/drive/1QCERSMUjqGetGGujdrvv_6_EeoIcd_9M?usp=sharing
